# ALFA Context Memory: Snapshot-Based Project Continuity

Long-running AI work often spans many sessions, tools, repositories, and local notes. The context window eventually fills up, but the project still needs continuity.

This cookbook shows a practical memory loop: collect local source material, create compact snapshots, accumulate them over time, and inject the distilled memory into the next conversation. Grok can then use the injected memory as project context while still checking live files before acting.

## Workflow

1. Read folders, markdown files, and zip archives.
2. Skip common generated folders and secret files.
3. Create source cards with short excerpts and tags.
4. Save a timestamped snapshot.
5. Render a cumulative context injection for the next session.

The pattern is intentionally local-first. You can use it before calling an API, or pair it with Grok to synthesize sharper summaries.

In [ ]:
from pathlib import Path
from alfa_brain import build_snapshot, render_injection, write_snapshot, load_snapshots

MEMORY_DIR = Path("memory")
MEMORY_DIR.mkdir(exist_ok=True)

## Configure Sources

`sources.example.json` points at local folders and zip archives. In a public repo, keep private source paths and generated memory out of version control.

In [ ]:
import json

config = json.loads(Path("sources.example.json").read_text(encoding="utf-8"))
sources = [Path(item) for item in config["sources"]]
sources

## Create a Snapshot

A snapshot is a compact view of the current knowledge state. It is not meant to replace source files; it is a lightweight continuity layer.

In [ ]:
snapshot = build_snapshot(
    sources,
    max_file_bytes=250_000,
    max_excerpt_chars=1_400,
)

snapshot_path = write_snapshot(snapshot, MEMORY_DIR)
snapshot_path, snapshot.source_count, snapshot.top_terms[:10]

## Inspect Source Cards

Source cards give the next session just enough signal to understand what exists and where it came from.

In [ ]:
for card in snapshot.source_cards[:5]:
    print(card.title)
    print(card.source)
    print(", ".join(card.tags))
    print(card.excerpt[:220])
    print("---")

## Render Context Injection

The injection is the text block to paste at the start of the next AI session. It uses XML-style sections so the model can distinguish memory, sources, and rules.

In [ ]:
snapshots = load_snapshots(MEMORY_DIR)
injection = render_injection(snapshots)

injection_path = MEMORY_DIR / "context-injection.md"
injection_path.write_text(injection, encoding="utf-8")

print(injection[:2_000])

## Optional: Ask Grok to Compress the Injection

When the cumulative injection grows too large, use Grok to compress it into a smaller project memory. Keep the raw snapshots locally, then inject only the compressed memory.

In [ ]:
# %pip install --quiet openai python-dotenv
# import os
# from dotenv import load_dotenv
# from openai import OpenAI
#
# load_dotenv()
# client = OpenAI(base_url="https://api.x.ai/v1", api_key=os.getenv("XAI_API_KEY"))
#
# compression_prompt = f"""
# Compress this ALFA Brain context injection into a durable project memory.
# Keep stable facts, active goals, repo structure, and next actions.
# Remove repetition, raw excerpts, and anything that looks like a secret.
#
# {injection}
# """
#
# response = client.chat.completions.create(
#     model="grok-4",
#     messages=[{"role": "user", "content": compression_prompt}],
# )
# print(response.choices[0].message.content)

## Operating Rule

At the end of a long conversation, create a fresh snapshot and render a new injection. At the start of the next conversation, paste the latest injection before asking for new work.

This gives the system a simple rhythm: **conversation -> snapshot -> cumulative memory -> next-session injection**.